# Single_label_all
Make the data Single label. Treat SP and ID as non-hybrid.

In [2]:
import sqlite3
from pathlib import Path


# Path to the specific .db file
DB_PATH = Path(
    r"C:\Users\alrazz\Documents\Hybrid SP_ID annotation\Combined_single.db"
)


def process_database(db_path):
    print(f"Processing: {db_path}")

    if not db_path.exists():
        print(f"ERROR: Database file does not exist:\n{db_path}")
        return

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute("""
        SELECT rowid, "Turku_NLP", "Turku_NLP_sub"
        FROM "texts"
    """)

    rows = cursor.fetchall()

    changed = 0

    for rowid, turku_nlp, turku_nlp_sub in rows:

        # Skip rows where Turku_NLP is NULL
        if turku_nlp is None:
            continue

        # Split both columns
        nlp_parts = [x.strip() for x in turku_nlp.split(";")]

        if turku_nlp_sub is not None:
            sub_parts = [x.strip() for x in turku_nlp_sub.split(";")]
        else:
            sub_parts = []

        # Find the first occurrence of ID or SP
        cut_position = None

        for i, value in enumerate(nlp_parts):

            if value in ("ID", "SP"):

                # Keep the ID/SP itself
                cut_position = i + 1

                # If the next item is also ID or SP,
                # keep that one too.
                if (
                    i + 1 < len(nlp_parts)
                    and nlp_parts[i + 1] in ("ID", "SP")
                ):
                    cut_position = i + 2

                break

        # No ID or SP found
        if cut_position is None:
            continue

        # Truncate both columns at exactly the same position
        new_nlp_parts = nlp_parts[:cut_position]
        new_sub_parts = sub_parts[:cut_position]

        # Join them back together
        new_nlp = " ; ".join(new_nlp_parts)
        new_sub = " ; ".join(new_sub_parts)

        # Update database
        cursor.execute("""
            UPDATE "texts"
            SET "Turku_NLP" = ?,
                "Turku_NLP_sub" = ?
            WHERE rowid = ?
        """, (new_nlp, new_sub, rowid))

        changed += 1

    conn.commit()
    conn.close()

    print(f"Changed {changed} rows.")
    print("Done.")


if __name__ == "__main__":
    process_database(DB_PATH)

Processing: C:\Users\alrazz\Documents\Hybrid SP_ID annotation\Combined_single.db
Changed 796 rows.
Done.


# Keep SP hybrid
Keep SP hybrid while ID single

In [6]:
import sqlite3


# Specific database file
DB_FILE = r"C:\Users\alrazz\Documents\Hybrid SP_ID annotation\Combined_hybrid - Copy.db"


def truncate_row(turku_nlp, turku_nlp_sub):
    """
    Truncate Turku_NLP according to these rules:

    1. If there is no ID -> leave unchanged.
    2. Normally, truncate immediately after ID.
    3. If SP immediately follows ID, keep both ID and SP.

    Turku_NLP_sub is truncated at the same position.
    """

    if not turku_nlp:
        return turku_nlp, turku_nlp_sub

    # Split Turku_NLP into parts
    nlp_parts = [x.strip() for x in turku_nlp.split(";")]

    # If ID does not exist, don't change anything
    if "ID" not in nlp_parts:
        return turku_nlp, turku_nlp_sub

    # Find the first ID
    id_index = nlp_parts.index("ID")

    # Normally keep through ID
    cutoff_index = id_index

    # Exception:
    # If SP immediately follows ID, also keep SP
    if id_index + 1 < len(nlp_parts) and nlp_parts[id_index + 1] == "SP":
        cutoff_index = id_index + 1

    # Keep everything up to the cutoff
    new_nlp_parts = nlp_parts[:cutoff_index + 1]

    # Apply exactly the same cutoff to Turku_NLP_sub
    if turku_nlp_sub:
        sub_parts = [x.strip() for x in turku_nlp_sub.split(";")]
        new_sub_parts = sub_parts[:cutoff_index + 1]
    else:
        new_sub_parts = []

    # Reconstruct strings
    new_nlp = " ; ".join(new_nlp_parts)
    new_sub = " ; ".join(new_sub_parts)

    return new_nlp, new_sub


# Connect to the database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Read rows
cursor.execute("""
    SELECT rowid, "Turku_NLP", "Turku_NLP_sub"
    FROM "texts"
""")

rows = cursor.fetchall()

changed = 0

for rowid, turku_nlp, turku_nlp_sub in rows:

    new_nlp, new_sub = truncate_row(
        turku_nlp,
        turku_nlp_sub
    )

    # Only update if something actually changed
    if new_nlp != turku_nlp or new_sub != turku_nlp_sub:

        cursor.execute("""
            UPDATE "texts"
            SET "Turku_NLP" = ?,
                "Turku_NLP_sub" = ?
            WHERE rowid = ?
        """, (new_nlp, new_sub, rowid))

        changed += 1


# Save changes
conn.commit()
conn.close()

print(f"Done. Rows changed: {changed}")

Done. Rows changed: 330


# Keep ID hybrid
Keep ID hybrid while SP single

In [8]:
import sqlite3


# Specific database file
DB_FILE = r"C:\Users\alrazz\Documents\Hybrid SP_ID annotation\Combined_ID_hybrid.db"


def truncate_row(turku_nlp, turku_nlp_sub):

    if not turku_nlp:
        return turku_nlp, turku_nlp_sub

    # Split Turku_NLP into parts
    nlp_parts = [x.strip() for x in turku_nlp.split(";")]

    # If ID does not exist, don't change anything
    if "SP" not in nlp_parts:
        return turku_nlp, turku_nlp_sub

    # Find the first ID
    id_index = nlp_parts.index("SP")

    # Normally keep through ID
    cutoff_index = id_index

    # Exception:
    # If SP immediately follows ID, also keep SP
    if id_index + 1 < len(nlp_parts) and nlp_parts[id_index + 1] == "ID":
        cutoff_index = id_index + 1

    # Keep everything up to the cutoff
    new_nlp_parts = nlp_parts[:cutoff_index + 1]

    # Apply exactly the same cutoff to Turku_NLP_sub
    if turku_nlp_sub:
        sub_parts = [x.strip() for x in turku_nlp_sub.split(";")]
        new_sub_parts = sub_parts[:cutoff_index + 1]
    else:
        new_sub_parts = []

    # Reconstruct strings
    new_nlp = " ; ".join(new_nlp_parts)
    new_sub = " ; ".join(new_sub_parts)

    return new_nlp, new_sub


# Connect to the database
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

# Read rows
cursor.execute("""
    SELECT rowid, "Turku_NLP", "Turku_NLP_sub"
    FROM "texts"
""")

rows = cursor.fetchall()

changed = 0

for rowid, turku_nlp, turku_nlp_sub in rows:

    new_nlp, new_sub = truncate_row(
        turku_nlp,
        turku_nlp_sub
    )

    # Only update if something actually changed
    if new_nlp != turku_nlp or new_sub != turku_nlp_sub:

        cursor.execute("""
            UPDATE "texts"
            SET "Turku_NLP" = ?,
                "Turku_NLP_sub" = ?
            WHERE rowid = ?
        """, (new_nlp, new_sub, rowid))

        changed += 1


# Save changes
conn.commit()
conn.close()

print(f"Done. Rows changed: {changed}")

Done. Rows changed: 381


# Handle SP-ID combination

Have to handle instances that ID and SP are together manually for hybrid